In [10]:
import scanpy as sc
import pandas as pd

group_files = {
    # "post_nres_ASDC":        "./adatas/adata_post_nres_ASDC.h5ad",
    # "post_nres_MPPpreB":     "./adatas/adata_post_nres_MPPpreB.h5ad",
    # "post_nres_HSCLSC":      "./adatas/adata_post_nres_HSCLSC.h5ad",
    # "post_nres_MPPCLP1":     "./adatas/adata_post_nres_MPPCLP1.h5ad",
    # "post_nres_EMP":         "./adatas/adata_post_nres_EMP.h5ad",
    # "post_nres_Macrophage1": "./adatas/adata_post_nres_Macrophage1.h5ad",
    # "post_nres_LE":          "./adatas/adata_post_nres_LE.h5ad",
    # "post_nres_GMP1":        "./adatas/adata_post_nres_GMP1.h5ad",
    # "post_nres_Erythroid":   "./adatas/adata_post_nres_Erythroid.h5ad",
}
group_files = {
    "pre_nres_HSCLSC":      "./adatas/adata_pre_nres_HSCLSClow.h5ad",
    "pre_nres_HSCLSC-M":      "./adatas/adata_pre_nres_HSCLSChigh.h5ad",
    "pre_nres_MPPCLP1":     "./adatas/adata_pre_nres_MPPCLP1.h5ad",
}


rows = []

for ct, f in group_files.items():
    adata = sc.read_h5ad(f)
    tmp = (
        adata.obs
        # .groupby("patient")
        .groupby("patient", observed=False)
        .size()
        .reset_index(name="n_cells")
    )
    tmp["cell_type"] = ct
    rows.append(tmp)

df_counts = pd.concat(rows, ignore_index=True)


In [13]:
import scanpy as sc
import pandas as pd

group_files = {
    # "post_nres_ASDC":        "./adatas/adata_post_nres_ASDC.h5ad",
    # "post_nres_MPPpreB":     "./adatas/adata_post_nres_MPPpreB.h5ad",
    # "post_nres_HSCLSC":      "./adatas/adata_post_nres_HSCLSC.h5ad",
    # "post_nres_MPPCLP1":     "./adatas/adata_post_nres_MPPCLP1.h5ad",
    # "post_nres_EMP":         "./adatas/adata_post_nres_EMP.h5ad",
    # "post_nres_Macrophage1": "./adatas/adata_post_nres_Macrophage1.h5ad",
    # "post_nres_LE":          "./adatas/adata_post_nres_LE.h5ad",
    # "post_nres_GMP1":        "./adatas/adata_post_nres_GMP1.h5ad",
    # "post_nres_Erythroid":   "./adatas/adata_post_nres_Erythroid.h5ad",
}
group_files = {
    "pre_nres_HSCLSC":      "./adatas/RR/adata_pre_nres_HSCLSClow.h5ad",
    "pre_nres_HSCLSC-M":      "./adatas/RR/adata_pre_nres_HSCLSChigh.h5ad",
    "pre_nres_MPPCLP1":     "./adatas/RR/adata_pre_nres_MPPCLP1.h5ad",
}


rows = []

for ct, f in group_files.items():
    adata = sc.read_h5ad(f)
    tmp = (
        adata.obs
        # .groupby("patient")
        .groupby("patient", observed=False)
        .size()
        .reset_index(name="n_cells")
    )
    tmp["cell_type"] = ct
    rows.append(tmp)

df_counts = pd.concat(rows, ignore_index=True)


FileNotFoundError: [Errno 2] Unable to synchronously open file (unable to open file: name = './adatas/RR/adata_pre_nres_HSCLSClow.h5ad', errno = 2, error message = 'No such file or directory', flags = 0, o_flags = 0)

In [11]:
df_counts["total_cells"] = (
    df_counts
    .groupby("cell_type")["n_cells"]
    .transform("sum")
)

df_counts["patient_ratio"] = (
    df_counts["n_cells"] / df_counts["total_cells"]
)

df_counts


,patient,n_cells,cell_type,total_cells,patient_ratio
0,Pt 7,23,pre_nres_HSCLSC,227,0.101322
1,Pt 15,98,pre_nres_HSCLSC,227,0.431718
2,Pt 17,106,pre_nres_HSCLSC,227,0.466960
3,Pt 7,2,pre_nres_HSCLSC-M,350,0.005714
4,Pt 15,34,pre_nres_HSCLSC-M,350,0.097143
5,Pt 17,314,pre_nres_HSCLSC-M,350,0.897143
6,Pt 7,13,pre_nres_MPPCLP1,228,0.057018
7,Pt 15,11,pre_nres_MPPCLP1,228,0.048246
8,Pt 17,204,pre_nres_MPPCLP1,228,0.894737


In [12]:
# target_cts = [
#     "post_nres_ASDC",
#     "post_nres_MPPpreB",
#     "post_nres_MPPCLP1"
# ]
target_cts = [
    "pre_nres_HSCLSC",
    "pre_nres_HSCLSC-M",
    "pre_nres_MPPCLP1"
]

df_counts["group_class"] = df_counts["cell_type"].apply(
    lambda x: "target" if x in target_cts else "other"
)
patient_matrix = (
    df_counts
    .pivot_table(
        index="cell_type",
        columns="patient",
        values="patient_ratio",
        fill_value=0
    )
)

patient_matrix


C:\Users\abdul\AppData\Local\Temp\ipykernel_28948\3609058200.py:16: FutureWarning: The default value of observed=False is deprecated and will change to observed=True in a future version of pandas. Specify observed=False to silence this warning and retain the current behavior
  df_counts


patient,Pt 7,Pt 15,Pt 17
cell_type,,,
pre_nres_HSCLSC,0.101322,0.431718,0.466960
pre_nres_HSCLSC-M,0.005714,0.097143,0.897143
pre_nres_MPPCLP1,0.057018,0.048246,0.894737


In [ ]:
import numpy as np
from scipy.spatial.distance import jensenshannon

def js_dist(a, b):
    return jensenshannon(a, b)

dist_results = []

for ct in target_cts:
    v_ct = patient_matrix.loc[ct]
    v_other = patient_matrix.drop(ct).mean(axis=0)
    dist = js_dist(v_ct, v_other)
    dist_results.append((ct, dist))

pd.DataFrame(dist_results, columns=["cell_type", "JS_distance"])


In [ ]:
from scipy.stats import chi2_contingency

stats = []

for ct in target_cts:
    sub = df_counts.copy()
    sub["is_ct"] = sub["cell_type"] == ct

    table = (
        sub
        .pivot_table(
            index="patient",
            columns="is_ct",
            values="n_cells",
            aggfunc="sum",
            fill_value=0
        )
    )

    chi2, p, _, _ = chi2_contingency(table)
    stats.append((ct, p))

pd.DataFrame(stats, columns=["cell_type", "chi2_pvalue"])


In [ ]:
import pandas as pd

groups_to_check = [
    "post_nres_ASDC",
    "post_nres_MPPpreB",
    "post_nres_MPPCLP1"
]

patient_dist = []

for g in groups_to_check:
    adata = adatas[g]
    
    df = (
        adata.obs
        .groupby("patient")
        .size()
        .reset_index(name="n_cells")
    )
    df["group"] = g
    patient_dist.append(df)

patient_dist_df = pd.concat(patient_dist, ignore_index=True)
patient_dist_df
